In [1]:
import argparse
import json
import csv
import glob

import os
import sys
sys.path.append("../..")
from pathlib import Path
from tqdm import tqdm
import pandas as pd
import numpy as np
import torch

import random
import monai
from monai.data import CacheDataset, DataLoader
import monai.transforms as transforms

import nibabel as nib

#import seaborn as sns
import matplotlib.pyplot as plt

from multiprocessing import Pool
from functools import partial

from monai.metrics import compute_iou, DiceMetric

/home/fehrdelt/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


In [2]:
#ROOT_DIR = "/bettik/PROJECTS/pr-gin5_aini/fehrdelt/"
ROOT_DIR = "/home/fehrdelt/bettik/"

In [3]:
adc_anomaly_maps_select_params_folder = f"{ROOT_DIR}datasets/anomaly_maps/exp_2_2_select_params/large/"
adc_anomaly_maps_folder = f"{ROOT_DIR}datasets/anomaly_maps/exp_2_2/large/"

flair_anomaly_maps_select_params_folder = f"{ROOT_DIR}datasets/anomaly_maps/exp_3_2_select_params/large/"
flair_anomaly_maps_folder = f"{ROOT_DIR}datasets/anomaly_maps/exp_3_2/large/"

combined_anomaly_maps_select_params_folder = f"{ROOT_DIR}datasets/anomaly_maps/combined_ano_maps_select_params/"
combined_anomaly_maps_folder = f"{ROOT_DIR}datasets/anomaly_maps/combined_ano_maps/"

combined_masks_folder = f"{ROOT_DIR}datasets/final_soop_dataset_small/masks_combined_registered/"


In [4]:
adc_ano_maps_select_params = os.listdir(adc_anomaly_maps_select_params_folder)
adc_ano_maps_select_params = sorted([f for f in adc_ano_maps_select_params if "_t_100" in f])

adc_ano_maps = os.listdir(adc_anomaly_maps_folder)
adc_ano_maps = sorted([f for f in adc_ano_maps if "seg" not in f])


flair_ano_maps_select_params = os.listdir(flair_anomaly_maps_select_params_folder)
flair_ano_maps_select_params = sorted([f for f in flair_ano_maps_select_params if "_t_150" in f])

flair_ano_maps = os.listdir(flair_anomaly_maps_folder)
flair_ano_maps = sorted([f for f in flair_ano_maps if "seg" not in f])


combined_ano_maps_select_params = os.listdir(combined_anomaly_maps_select_params_folder)
combined_ano_maps_select_params = sorted([f for f in combined_ano_maps_select_params if "seg" not in f])

combined_ano_maps = os.listdir(combined_anomaly_maps_folder)
combined_ano_maps = sorted([f for f in combined_ano_maps if "seg" not in f])


combined_masks_select_params = sorted([path for path in combined_ano_maps_select_params])

combined_masks = sorted([path for path in combined_ano_maps])

In [5]:
print("ADC ano maps select params:      ", adc_ano_maps_select_params)
print("\nADC ano maps:                    ", adc_ano_maps)
print("\nFLAIR ano maps select params:    ", flair_ano_maps_select_params)
print("\nFLAIR ano maps:                  ", flair_ano_maps)
print("\nCombined ano maps select params: ", combined_ano_maps_select_params)
print("\nCombined ano maps:               ", combined_ano_maps)
print("\nCombined masks select_params:    ", combined_masks_select_params)
print("\nCombined masks:                  ", combined_masks)

ADC ano maps select params:       ['sub-1010_t_100.nii.gz', 'sub-1015_t_100.nii.gz', 'sub-1164_t_100.nii.gz', 'sub-1204_t_100.nii.gz', 'sub-1209_t_100.nii.gz', 'sub-1213_t_100.nii.gz', 'sub-1227_t_100.nii.gz', 'sub-1246_t_100.nii.gz', 'sub-1258_t_100.nii.gz', 'sub-127_t_100.nii.gz', 'sub-1323_t_100.nii.gz', 'sub-1354_t_100.nii.gz', 'sub-1358_t_100.nii.gz', 'sub-1373_t_100.nii.gz', 'sub-1395_t_100.nii.gz', 'sub-1410_t_100.nii.gz', 'sub-1422_t_100.nii.gz', 'sub-1432_t_100.nii.gz', 'sub-1445_t_100.nii.gz', 'sub-1478_t_100.nii.gz', 'sub-1480_t_100.nii.gz', 'sub-1485_t_100.nii.gz', 'sub-1488_t_100.nii.gz', 'sub-1508_t_100.nii.gz', 'sub-1555_t_100.nii.gz', 'sub-1569_t_100.nii.gz', 'sub-1656_t_100.nii.gz', 'sub-1670_t_100.nii.gz', 'sub-1719_t_100.nii.gz', 'sub-1725_t_100.nii.gz', 'sub-1736_t_100.nii.gz', 'sub-190_t_100.nii.gz', 'sub-198_t_100.nii.gz', 'sub-241_t_100.nii.gz', 'sub-247_t_100.nii.gz', 'sub-278_t_100.nii.gz', 'sub-294_t_100.nii.gz', 'sub-303_t_100.nii.gz', 'sub-321_t_100.nii.gz',

In [6]:
combined_masks_folder = f"{ROOT_DIR}datasets/final_soop_dataset_small/masks_combined_registered/"

In [7]:
combined_masks_paths = os.listdir(combined_masks_folder)  # Assuming same names for chronic and acute masks

In [8]:
print("combined masks:", combined_masks_paths)

combined masks: ['sub-1674.nii.gz', 'sub-1489.nii.gz', 'sub-1581.nii.gz', 'sub-552.nii.gz', 'sub-603.nii.gz', 'sub-1093.nii.gz', 'sub-1033.nii.gz', 'sub-857.nii.gz', 'sub-345.nii.gz', 'sub-549.nii.gz', 'sub-35.nii.gz', 'sub-943.nii.gz', 'sub-1021.nii.gz', 'sub-311.nii.gz', 'sub-560.nii.gz', 'sub-816.nii.gz', 'sub-1151.nii.gz', 'sub-1374.nii.gz', 'sub-822.nii.gz', 'sub-901.nii.gz', 'sub-173.nii.gz', 'sub-1072.nii.gz', 'sub-825.nii.gz', 'sub-449.nii.gz', 'sub-1213.nii.gz', 'sub-1628.nii.gz', 'sub-1277.nii.gz', 'sub-617.nii.gz', 'sub-194.nii.gz', 'sub-1600.nii.gz', 'sub-1361.nii.gz', 'sub-1698.nii.gz', 'sub-790.nii.gz', 'sub-1111.nii.gz', 'sub-1046.nii.gz', 'sub-1637.nii.gz', 'sub-1017.nii.gz', 'sub-1149.nii.gz', 'sub-1041.nii.gz', 'sub-128.nii.gz', 'sub-1516.nii.gz', 'sub-1559.nii.gz', 'sub-1105.nii.gz', 'sub-884.nii.gz', 'sub-138.nii.gz', 'sub-328.nii.gz', 'sub-1085.nii.gz', 'sub-1432.nii.gz', 'sub-1330.nii.gz', 'sub-503.nii.gz', 'sub-743.nii.gz', 'sub-1617.nii.gz', 'sub-354.nii.gz', 's

In [9]:
patient_ids_to_test = [os.path.basename(path).split(".")[0] for path in combined_ano_maps]
print(patient_ids_to_test)

['sub-366', 'sub-370', 'sub-374', 'sub-398', 'sub-400', 'sub-422', 'sub-433', 'sub-443', 'sub-446', 'sub-457', 'sub-463', 'sub-466', 'sub-525', 'sub-53', 'sub-530', 'sub-543', 'sub-56', 'sub-572', 'sub-631', 'sub-634', 'sub-651', 'sub-661', 'sub-692', 'sub-698', 'sub-724', 'sub-751', 'sub-760', 'sub-761', 'sub-768', 'sub-776', 'sub-791', 'sub-8', 'sub-803', 'sub-866', 'sub-877', 'sub-937', 'sub-942', 'sub-946', 'sub-959', 'sub-960']


#### Combined (ADC + FLAIR)

In [26]:
# Masks

combined_masks_transforms = transforms.Compose([
    transforms.LoadImage(),
    transforms.EnsureChannelFirst(),
])


combined_masks_dataset_select_params = CacheDataset(data=[combined_masks_folder + file for file in combined_ano_maps_select_params], transform=combined_masks_transforms)
combined_masks_dataloader_select_params = DataLoader(combined_masks_dataset_select_params, batch_size=2, shuffle=False, num_workers=0)

combined_masks_dataset = CacheDataset(data=[combined_masks_folder + file for file in combined_ano_maps], transform=combined_masks_transforms)
combined_masks_dataloader = DataLoader(combined_masks_dataset, batch_size=2, shuffle=False, num_workers=0)

Loading dataset: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 40/40 [00:01<00:00, 26.92it/s]


In [27]:
combined_maps_transforms = transforms.Compose([
    transforms.LoadImage(),
    transforms.EnsureChannelFirst(),
])


combined_maps_dataset_select_params = CacheDataset(data=[os.path.join(combined_anomaly_maps_select_params_folder, path) for path in combined_ano_maps_select_params], transform=combined_maps_transforms)
combined_maps_dataloader_select_params = DataLoader(combined_maps_dataset_select_params, batch_size=2, shuffle=False, num_workers=0)

combined_maps_dataset = CacheDataset(data=[os.path.join(combined_anomaly_maps_folder, path) for path in combined_ano_maps], transform=combined_maps_transforms)
combined_maps_dataloader = DataLoader(combined_maps_dataset, batch_size=2, shuffle=False, num_workers=0)

Loading dataset: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 40/40 [00:26<00:00,  1.53it/s]


In [28]:
print(f"combined_maps_dataset_select_params: {[os.path.join(combined_anomaly_maps_select_params_folder, path) for path in combined_ano_maps_select_params]}")
print(f"\ncombined_masks_dataset_select_params: {[combined_masks_folder + file for file in combined_ano_maps_select_params]}")

combined_maps_dataset_select_params: ['/home/fehrdelt/bettik/datasets/anomaly_maps/combined_ano_maps_select_params/sub-1010.nii.gz', '/home/fehrdelt/bettik/datasets/anomaly_maps/combined_ano_maps_select_params/sub-1015.nii.gz', '/home/fehrdelt/bettik/datasets/anomaly_maps/combined_ano_maps_select_params/sub-1164.nii.gz', '/home/fehrdelt/bettik/datasets/anomaly_maps/combined_ano_maps_select_params/sub-1204.nii.gz', '/home/fehrdelt/bettik/datasets/anomaly_maps/combined_ano_maps_select_params/sub-1209.nii.gz', '/home/fehrdelt/bettik/datasets/anomaly_maps/combined_ano_maps_select_params/sub-1213.nii.gz', '/home/fehrdelt/bettik/datasets/anomaly_maps/combined_ano_maps_select_params/sub-1227.nii.gz', '/home/fehrdelt/bettik/datasets/anomaly_maps/combined_ano_maps_select_params/sub-1246.nii.gz', '/home/fehrdelt/bettik/datasets/anomaly_maps/combined_ano_maps_select_params/sub-1258.nii.gz', '/home/fehrdelt/bettik/datasets/anomaly_maps/combined_ano_maps_select_params/sub-127.nii.gz', '/home/fehrde

In [29]:
dm = DiceMetric(reduction="sum")

In [30]:
thresholds_to_try = np.linspace(0.01, 0.1, 60)

**find the best threshold on the select params dataset**

In [54]:
dice_scores_combined_select_params = {}

for threshold in tqdm(thresholds_to_try):

    this_dice_score = []

    for ano_map, ano_mask in zip(combined_maps_dataloader_select_params, combined_masks_dataloader_select_params):
                
        ano_mask[ano_mask > 0.5] = 1

        combined_segmentation = (ano_map > threshold)

        combined_dice_score = dm(combined_segmentation, ano_mask).flatten()
        
        combined_dice_score = combined_dice_score[~np.isnan(combined_dice_score)]
        
        this_dice_score.append(combined_dice_score)            
        
    dice_scores_combined_select_params[threshold] = np.mean(this_dice_score)

    

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 60/60 [00:42<00:00,  1.40it/s]


In [55]:
best_threshold_combined = max(dice_scores_combined_select_params, key=dice_scores_combined_select_params.get)
print(f"Best Dice Score combined ano maps: {dice_scores_combined_select_params[best_threshold_combined]} at threshold: {best_threshold_combined}")

Best Dice Score combined ano maps: 0.47802796959877014 at threshold: 0.07101694915254238


**Compute the final DICE score with the selected threshold**

In [56]:

final_dice_score_combined_ano_maps = []

for ano_map, ano_mask in tqdm(zip(combined_maps_dataloader, combined_masks_dataloader)):
            
    ano_mask[ano_mask > 0.5] = 1

    combined_segmentation = (ano_map > best_threshold_combined)

    combined_dice_score = dm(combined_segmentation, ano_mask).flatten()
    
    combined_dice_score = combined_dice_score[~np.isnan(combined_dice_score)]
    
    final_dice_score_combined_ano_maps.append(combined_dice_score)
    

print(f"Final Dice Score combined ano maps {np.mean(final_dice_score_combined_ano_maps)} with {best_threshold_combined} best threshold")

    

20it [00:00, 48.60it/s]

Final Dice Score combined ano maps 0.6571252942085266 with 0.07101694915254238 best threshold


**Compute the 95% confidence intervals**

In [59]:
rng = np.random.default_rng(42)

# Flatten the final_dice_score list to get individual dice scores
final_dice_scores_combined_ano_maps_flat = np.concatenate([score.numpy() for score in final_dice_score_combined_ano_maps])

idx = np.arange(len(final_dice_scores_combined_ano_maps_flat))

test_dice_combined_ano_maps = []

for i in range(200): 
    # bootstrap with 200 rounds: random sampling with replacement of the predictions
    pred_idx = rng.choice(idx, size=len(idx), replace=True)
    
    dice_boot = np.mean(final_dice_scores_combined_ano_maps_flat[pred_idx])
    
    test_dice_combined_ano_maps.append(dice_boot)

# Compute the mean and 95% confidence intervals
bootstrap_dice_test_mean_combined_ano_maps = np.mean(test_dice_combined_ano_maps)
ci_lower = np.percentile(test_dice_combined_ano_maps, 2.5)
ci_upper = np.percentile(test_dice_combined_ano_maps, 97.5)

print(f"Final combined ano maps DICE score: {bootstrap_dice_test_mean_combined_ano_maps:.2f} - 95% CI {ci_lower:.2f}-{ci_upper:.2f}")


Final combined ano maps DICE score: 0.66 - 95% CI 0.59-0.71


#### ADC only

In [43]:
adc_maps_transforms = transforms.Compose([
    transforms.LoadImage(),
    transforms.EnsureChannelFirst(),
])


adc_maps_dataset_select_params = CacheDataset(data=[os.path.join(adc_anomaly_maps_select_params_folder, path) for path in adc_ano_maps_select_params], transform=adc_maps_transforms)
adc_maps_dataloader_select_params = DataLoader(adc_maps_dataset_select_params, batch_size=2, shuffle=False, num_workers=0)

adc_maps_dataset = CacheDataset(data=[os.path.join(adc_anomaly_maps_folder, path) for path in adc_ano_maps], transform=adc_maps_transforms)
adc_maps_dataloader = DataLoader(adc_maps_dataset, batch_size=2, shuffle=False, num_workers=0)

Loading dataset: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 40/40 [00:16<00:00,  2.49it/s]


**find the best threshold on the select params dataset**

In [60]:
dice_scores_adc_select_params = {}

for threshold in tqdm(thresholds_to_try):

    this_dice_score = []

    for ano_map, ano_mask in zip(adc_maps_dataloader_select_params, combined_masks_dataloader_select_params):
                
        ano_mask[ano_mask > 0.5] = 1

        adc_segmentation = (ano_map > threshold)

        adc_dice_score = dm(adc_segmentation, ano_mask).flatten()
        
        adc_dice_score = adc_dice_score[~np.isnan(adc_dice_score)]
        
        this_dice_score.append(adc_dice_score)            
        
    dice_scores_adc_select_params[threshold] = np.mean(this_dice_score)
    

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 60/60 [00:45<00:00,  1.32it/s]


In [62]:
best_threshold_adc = max(dice_scores_adc_select_params, key=dice_scores_adc_select_params.get)
print(f"Best Dice Score ADC: {dice_scores_adc_select_params[best_threshold_adc]} at threshold: {best_threshold_adc}")

Best Dice Score ADC: 0.34980419278144836 at threshold: 0.08474576271186442


**Compute the final DICE score with the selected threshold**

In [63]:

final_dice_score_adc = []

for ano_map, ano_mask in tqdm(zip(adc_maps_dataloader, combined_masks_dataloader)):
            
    ano_mask[ano_mask > 0.5] = 1

    adc_segmentation = (ano_map > best_threshold_adc)

    adc_dice_score = dm(adc_segmentation, ano_mask).flatten()
    
    adc_dice_score = adc_dice_score[~np.isnan(adc_dice_score)]
    
    final_dice_score_adc.append(adc_dice_score)
    

print(f"Final Dice Score ADC {np.mean(final_dice_score_adc)} with {best_threshold_adc} best threshold")

    

20it [00:00, 41.21it/s]

Final Dice Score ADC 0.5356118083000183 with 0.08474576271186442 best threshold


**Compute the 95% confidence intervals**

In [64]:
rng = np.random.default_rng(42)

# Flatten the final_dice_score list to get individual dice scores
final_dice_scores_adc_ano_maps_flat = np.concatenate([score.numpy() for score in final_dice_score_adc])

idx = np.arange(len(final_dice_scores_adc_ano_maps_flat))

test_dice_adc_ano_maps = []

for i in range(200): 
    # bootstrap with 200 rounds: random sampling with replacement of the predictions
    pred_idx = rng.choice(idx, size=len(idx), replace=True)
    
    dice_boot = np.mean(final_dice_scores_adc_ano_maps_flat[pred_idx])
    
    test_dice_adc_ano_maps.append(dice_boot)

# Compute the mean and 95% confidence intervals
bootstrap_dice_test_mean_adc_ano_maps = np.mean(test_dice_adc_ano_maps)
ci_lower = np.percentile(test_dice_adc_ano_maps, 2.5)
ci_upper = np.percentile(test_dice_adc_ano_maps, 97.5)

print(f"Final adc ano maps DICE score: {bootstrap_dice_test_mean_adc_ano_maps:.2f} - 95% CI {ci_lower:.2f}-{ci_upper:.2f}")


Final adc ano maps DICE score: 0.54 - 95% CI 0.46-0.61


#### FLAIR only

In [47]:
flair_maps_transforms = transforms.Compose([
    transforms.LoadImage(),
    transforms.EnsureChannelFirst(),
])


flair_maps_dataset_select_params = CacheDataset(data=[os.path.join(flair_anomaly_maps_select_params_folder, path) for path in flair_ano_maps_select_params], transform=flair_maps_transforms)
flair_maps_dataloader_select_params = DataLoader(flair_maps_dataset_select_params, batch_size=2, shuffle=False, num_workers=0)

flair_maps_dataset = CacheDataset(data=[os.path.join(flair_anomaly_maps_folder, path) for path in flair_ano_maps], transform=flair_maps_transforms)
flair_maps_dataloader = DataLoader(flair_maps_dataset, batch_size=2, shuffle=False, num_workers=0)

Loading dataset: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 40/40 [00:21<00:00,  1.88it/s]


**find the best threshold on the select params dataset**

In [65]:
dice_scores_flair_select_params = {}

for threshold in tqdm(thresholds_to_try):

    this_dice_score = []

    for ano_map, ano_mask in zip(flair_maps_dataloader_select_params, combined_masks_dataloader_select_params):
                
        ano_mask[ano_mask > 0.5] = 1

        flair_segmentation = (ano_map > threshold)

        flair_dice_score = dm(flair_segmentation, ano_mask).flatten()
        
        flair_dice_score = flair_dice_score[~np.isnan(flair_dice_score)]
        
        this_dice_score.append(flair_dice_score)            
        
    dice_scores_flair_select_params[threshold] = np.mean(this_dice_score)

    

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 60/60 [00:41<00:00,  1.46it/s]


In [66]:
best_threshold_flair = max(dice_scores_flair_select_params, key=dice_scores_flair_select_params.get)
print(f"Best Dice Score: {dice_scores_flair_select_params[best_threshold_flair]} at threshold: {best_threshold_flair}")

Best Dice Score: 0.37737923860549927 at threshold: 0.09542372881355933


**Compute the final DICE score with the selected threshold**

In [67]:

final_dice_score_flair = []

for ano_map, ano_mask in tqdm(zip(flair_maps_dataloader, combined_masks_dataloader)):
            
    ano_mask[ano_mask > 0.5] = 1

    flair_segmentation = (ano_map > best_threshold_flair)

    flair_dice_score = dm(flair_segmentation, ano_mask).flatten()
    
    flair_dice_score = flair_dice_score[~np.isnan(flair_dice_score)]
    
    final_dice_score_flair.append(flair_dice_score)
    

print(f"Final Dice Score {np.mean(final_dice_score_flair)} with {best_threshold_flair} best threshold")

    

20it [00:00, 29.83it/s]

Final Dice Score 0.3356262147426605 with 0.09542372881355933 best threshold


**Compute the 95% confidence intervals**

In [68]:
rng = np.random.default_rng(42)

# Flatten the final_dice_score list to get individual dice scores
final_dice_scores_flair_ano_maps_flat = np.concatenate([score.numpy() for score in final_dice_score_flair])

idx = np.arange(len(final_dice_scores_flair_ano_maps_flat))

test_dice_flair_ano_maps = []

for i in range(200): 
    # bootstrap with 200 rounds: random sampling with replacement of the predictions
    pred_idx = rng.choice(idx, size=len(idx), replace=True)
    
    dice_boot = np.mean(final_dice_scores_flair_ano_maps_flat[pred_idx])
    
    test_dice_flair_ano_maps.append(dice_boot)

# Compute the mean and 95% confidence intervals
bootstrap_dice_test_mean_flair_ano_maps = np.mean(test_dice_flair_ano_maps)
ci_lower = np.percentile(test_dice_flair_ano_maps, 2.5)
ci_upper = np.percentile(test_dice_flair_ano_maps, 97.5)

print(f"Final flair ano maps DICE score: {bootstrap_dice_test_mean_flair_ano_maps:.2f} - 95% CI {ci_lower:.2f}-{ci_upper:.2f}")


Final flair ano maps DICE score: 0.34 - 95% CI 0.27-0.41
